# Pauli Path GPU — Max Pauli String Count Sweep

Tests `θ_x = π/4` across coefficient truncation thresholds:
- `1.000E-04`, `5.00E-05`, `2.50E-05`, `1.60E-05`, `1.00E-05`

Reports `<Z_62>` expectation value and `max(num_paulis_hist)` per threshold.

## 1. Imports & Setup

In [1]:
import numpy as np
import json
import os
import time

from deqart_internal import pauli_path_gpu
from bluequbit.library.helpers.hardware_connectivites import IBM_127_HEAVY_HEX_MAP
from qiskit import QuantumCircuit

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

ModuleNotFoundError: No module named 'deqart_internal'

## 2. Circuit & Observable Parameters

In [ ]:
# ── Circuit parameters ────────────────────────────────────────────────────────
num_qubits        = 127
num_trotter_steps = 20
rzz_angle         = -np.pi / 2
rx_angle          = np.pi / 4          # θ_x = π/4

# ── Truncation thresholds ─────────────────────────────────────────────────────
deltas = [1.000e-4, 5.00e-5, 2.50e-5, 1.60e-5, 1.00e-5]
delta_labels = ["1.000E-04", "5.00E-05", "2.50E-05", "1.60E-05", "1.00E-05"]

# ── Observable <Z_62> ─────────────────────────────────────────────────────────
# BQ big-endian ordering: qubit 0 = rightmost character
# Qubit 62 from the right → position (num_qubits - 1 - 62) = 64 from the left
pauli_str = "I" * (num_qubits - 1 - 62) + "Z" + "I" * 62
pauli_sum = [(pauli_str, 1.0)]

print(f"Circuit: {num_qubits} qubits, {num_trotter_steps} Trotter steps")
print(f"θ_x = π/4 = {rx_angle:.6f} rad")
print(f"Observable: Z_62")
print(f"Thresholds: {delta_labels}")

## 3. Build Trotter Circuit

In [ ]:
qc = QuantumCircuit(num_qubits)

for _ in range(num_trotter_steps):
    for edge in IBM_127_HEAVY_HEX_MAP:
        qc.rzz(rzz_angle, edge[0], edge[1])
    for i in range(num_qubits):
        qc.rx(rx_angle, i)

print(f"Circuit depth : {qc.depth()}")
print(f"Gate count    : {qc.size()}")
print(f"Circuit built ✓")

## 4. Run Sweep — Expectation Value + Max Pauli Strings

In [ ]:
records = []

print(f"{'Delta':<14} {'<Z_62>':<14} {'Max Paulis':<16} {'Runtime (s)':<12} Status")
print("-" * 70)

for delta, label in zip(deltas, delta_labels):
    t0 = time.time()
    try:
        exp_val, num_paulis_hist, gate_times, _, _ = pauli_path_gpu.get_expectation_value(
            qc,
            pauli_sum,
            truncation_threshold=delta,
            return_internal_stats=True,
        )

        elapsed      = time.time() - t0
        max_paulis   = max(num_paulis_hist)

        record = {
            "delta":           delta,
            "delta_label":     label,
            "expectation_value": float(np.real(exp_val)),
            "max_paulis":      int(max_paulis),
            "num_paulis_hist": [int(x) for x in num_paulis_hist],
            "gate_times_ms":   [float(x) for x in gate_times],
            "run_time_s":      elapsed,
            "rx_angle":        rx_angle,
            "num_qubits":      num_qubits,
            "num_trotter_steps": num_trotter_steps,
            "rzz_angle":       rzz_angle,
            "observable":      "Z_62",
            "status":          "ok",
        }
        records.append(record)

        print(f"{label:<14} {float(np.real(exp_val)):<14.6f} {max_paulis:<16,} {elapsed:<12.2f} ✓")

    except Exception as e:
        elapsed = time.time() - t0
        record = {
            "delta":       delta,
            "delta_label": label,
            "error":       str(e),
            "run_time_s":  elapsed,
            "status":      "error",
        }
        records.append(record)
        print(f"{label:<14} {'ERROR':<14} {'—':<16} {elapsed:<12.2f} ✗  {e}")

print("\nSweep complete.")

## 5. Save Results to JSONL

In [ ]:
OUTPUT_FILE = "data/fig8_pauli_count_test.jsonl"
os.makedirs("data", exist_ok=True)

with open(OUTPUT_FILE, "w") as f:
    f.write(f"# θ_x=π/4  num_qubits={num_qubits}  num_trotter_steps={num_trotter_steps}\n")
    for r in records:
        f.write(json.dumps(r) + "\n")

print(f"Results saved → {OUTPUT_FILE}")

## 6. Results Table

In [ ]:
ok_records = [r for r in records if r["status"] == "ok"]

print(f"\n{'='*65}")
print(f"{'RESULTS — θ_x = π/4':^65}")
print(f"{'='*65}")
print(f"{'δ (threshold)':<16} {'<Z_62>':<14} {'Max |P| strings':<20} {'Time (s)'}")
print("-" * 65)
for r in ok_records:
    print(f"{r['delta_label']:<16} {r['expectation_value']:<14.6f} {r['max_paulis']:<20,} {r['run_time_s']:.2f}")
print(f"{'='*65}")

## 7. Plots

In [ ]:
if not ok_records:
    print("No successful runs to plot.")
else:
    deltas_ok  = [r["delta"]            for r in ok_records]
    ev_ok      = [r["expectation_value"] for r in ok_records]
    paulis_ok  = [r["max_paulis"]        for r in ok_records]
    times_ok   = [r["run_time_s"]        for r in ok_records]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(r"Pauli Path GPU Sweep — $\theta_X = \pi/4$", fontsize=13, fontweight="bold")

    color = "#1B3FA0"
    mk    = "o"
    ms    = 8

    # ── Panel 1: <Z_62> vs delta ───────────────────────────────────────────────
    ax = axes[0]
    ax.plot(deltas_ok, ev_ok, color=color, marker=mk, markersize=ms, linewidth=1.8)
    ax.set_xscale("log")
    ax.invert_xaxis()
    ax.set_xlabel(r"Truncation threshold $\delta$", fontsize=11)
    ax.set_ylabel(r"$\langle Z_{62} \rangle$", fontsize=11)
    ax.set_title(r"Expectation value vs $\delta$", fontsize=11)
    ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    ax.grid(True, alpha=0.3)

    # ── Panel 2: Max Pauli strings vs delta ───────────────────────────────────
    ax = axes[1]
    ax.plot(deltas_ok, paulis_ok, color="#C93030", marker="s", markersize=ms, linewidth=1.8)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.invert_xaxis()
    ax.set_xlabel(r"Truncation threshold $\delta$", fontsize=11)
    ax.set_ylabel("Max # Pauli strings", fontsize=11)
    ax.set_title(r"Max Pauli strings vs $\delta$", fontsize=11)
    ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    ax.yaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    ax.grid(True, alpha=0.3)

    # ── Panel 3: Runtime vs delta ─────────────────────────────────────────────
    ax = axes[2]
    ax.plot(deltas_ok, times_ok, color="#2A9D8F", marker="^", markersize=ms, linewidth=1.8)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.invert_xaxis()
    ax.set_xlabel(r"Truncation threshold $\delta$", fontsize=11)
    ax.set_ylabel("Runtime (s)", fontsize=11)
    ax.set_title(r"Runtime vs $\delta$", fontsize=11)
    ax.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("fig8_pauli_count_test.png", dpi=180, bbox_inches="tight")
    print("Plot saved → fig8_pauli_count_test.png")
    plt.show()

## 8. Pauli Count History (per threshold)

In [ ]:
# Plot num_paulis_hist for each threshold — shows how the Pauli population
# grows and shrinks through the circuit layers.

if ok_records:
    fig, axes = plt.subplots(1, len(ok_records), figsize=(4 * len(ok_records), 4), sharey=False)
    if len(ok_records) == 1:
        axes = [axes]

    palette = ["#1B3FA0", "#E89B2D", "#C93030", "#2A9D8F", "#7B2D8B"]

    for ax, r, col in zip(axes, ok_records, palette):
        hist = r["num_paulis_hist"]
        ax.plot(range(len(hist)), hist, color=col, linewidth=1.4)
        ax.axhline(max(hist), color=col, linestyle="--", linewidth=0.9, alpha=0.7,
                   label=f"max = {max(hist):,}")
        ax.set_title(f"δ = {r['delta_label']}", fontsize=10)
        ax.set_xlabel("Gate layer", fontsize=9)
        ax.set_ylabel("# active Pauli strings", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.25)

    fig.suptitle(r"Pauli string population through circuit — $\theta_X = \pi/4$",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("fig8_paulis_history.png", dpi=180, bbox_inches="tight")
    print("History plot saved → fig8_paulis_history.png")
    plt.show()